# Projection of an Upscaled Spline
Let $f_{0}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f_{0}(x)=\sum_{k\in{\mathbb{Z}}}\,c_{0}[{k\bmod K_{0}}]\,\beta^{n_{0}}(x-\delta x_{0}-k)$ be the realization of a periodic random spline at nominal scale, with a specified period $K_{0},$ degree $n_{0},$ and delay $\delta_{0}.$ We display this spline in thick gray. Then, $f:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f(x)=\sum_{k\in{\mathbb{Z}}}\,c[{k\bmod K}]\,\beta^{n}(x-\delta x-k)$ is the spline of period $K=M\,K_{0},$ arbitrary degree $n,$ and arbitrary delay $\delta x$ that minimizes the mean-square criterion $J=\int_{0}^{M\,K_{0}}\,\left(f(x)-f_{0}(\frac{x}{M})\right)^{2}\,{\mathrm{d}}x,$ where $M\in{\mathbb{N}}+1$ is a positive integer magnification factor. We display $f$ in blue, with samples at the integers indicated by circles and stem lines, and knots shown as black dots. The boundaries of one period are highlighted in red.

We print the value of the integral (over one period) of the product between the residue $\left(f(\cdot)-f_{0}(\frac{\cdot}{M})\right)$ and the projection $f.$ For optimal spline coefficients $c,$ the scalar product is expected to vanish. This is precisely what happens, up to numerical accuracy.

In [ ]:
# Load the required libraries
from IPython.display import display
from IPython.display import Math
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import warnings

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay
max_magnif = 5 # Maximal magnification factor

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic spline
f0 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 0)

# Plot
def update_plot (
    period0 = 6,
    degree0 = 0,
    delay0 = 0.0,
    magnif = 2,
    degree = 3,
    delay = 0.0
):
    global f0

    # Update of the spline
    if f0.period != period0:
        f0 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period0),
            degree = f0.degree
        )
    f0.degree = degree0
    f0.delay = delay0

    # Upscaling and projection
    f = f0.upscaled_projected(magnification = magnif, degree = degree, delay = delay)

    # Dynamic range
    image = {f0.image(), f.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plots
    subplot = plt.subplots()
    # Plot of the spline being magnified
    f0.upscaled(magnification = magnif).plot(
        subplot,
        plotpoints = 200 + 1,
        plotrange = plotrange,
        curve_fmt = "#e0e0e0",
        curve_lw = 7.0,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )

    # Plot of the projected spline
    f.plot(subplot, plotpoints = 200 + 1, plotrange = plotrange)

    # Final display
    plt.show()

    # Scalar product
    def integrand (
        x
    ):
        return (f.at(x) - f0.at(x / magnif)) * f.at(x)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dj = sp.integrate.quad(
            integrand,
            0,
            magnif * period0,
            points = np.concatenate((np.multiply(magnif, f0.get_knots()), f.get_knots())),
            limit = f0.period + f.period + 1
        )
    # Integral
    display(Math(
        r"""
        \int_{{{}}}^{{{}}}\,
        \left(f(x)-f_{{0}}(\frac{{x}}{{{}}})\right)\,f(x)\,
        {{\mathrm{{d}}}}x={:.2E}
        """.format(0, f.period, magnif, dj[0])
    ))

# Interaction
widgets.interactive(
    update_plot,
    period0 = (1, max_period),
    degree0 = (0, max_degree),
    delay0 = (-max_delay, max_delay),
    magnif = (1, max_magnif),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay)
)
